# Music Annotation Project V7.8.1: Implementation-Equivalence Audit

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook tests whether the V7.8 independent-data builder reproduces the development pipeline used to generate the V7.5/V7.7 evidence. It is read-only with respect to all source data, caches and released validation files.

## Fail-closed principle

V7.8 independent validation is not interpretable unless the two implementations agree on the same development pages and masks. This audit therefore compares:

1. mask preparation and connected components;
2. synthetic observed-page construction;
3. frozen `geometry_t80` reconstruction;
4. preserve/remove/ambiguous and horizontal/vertical/compact routing maps;
5. component bounding boxes and identifiers;
6. all shared predictor fields;
7. structural truth labels and their underlying ink/edge metrics;
8. outside-mask immutability.

The audit does not tune either implementation. It writes discrepancy tables, difference images and a machine-readable verdict into a new timestamped output directory.

## Authoritative inputs

The notebook prefers existing V7.5 persistent caches and tables as the development reference. If the exact V7.7 development artefacts are available, set the override paths in the configuration cell. The V7.8 builder implementation is embedded explicitly in this audit and fingerprinted.

A `PASS` requires exact equality for discrete arrays and identifiers, plus configured numerical tolerances for floating-point predictors and metrics. Missing authoritative artefacts produce `NOT_TESTED`, never `PASS`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib


In [ ]:
import cv2, json, hashlib, math, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V77_ROOT=ROOT/'structural_risk_calibration_v7_7'
BUILDER_ROOT=ROOT/'independent_structural_validation_v7_8'
OUTPUT_ROOT=ROOT/'implementation_equivalence_audit_v7_8_1'

V75_RUN_OVERRIDE=None
V77_RUN_OVERRIDE=None
MAX_PAGES=None             # None audits every matched development page.
PAGE_ALLOWLIST=None         # e.g. ['Page002','Page003']
FLOAT_ABS_TOLERANCE=1e-9
FLOAT_REL_TOLERANCE=1e-7
IMAGE_DIFFERENCE_TOLERANCE=0
MIN_COMPONENT_AREA=20

# Frozen V7.8-builder settings to audit.
GEOMETRY_THRESHOLD=80
CONTEXT_MARGIN=24
DARK_THRESHOLD=190
EDGE_LOW=70
EDGE_HIGH=170
HORIZONTAL_KERNEL=35
BUILDER_STRUCTURAL_THRESHOLDS={'ink_recall_min':0.80,'ink_f1_min':0.80,'edge_f1_min':0.75}

PREDICTOR_FIELDS=['w','h','area','aspect_ratio','fill_ratio','touches_page_border','context_pixels','context_edge_density','context_dark_density','context_mean_intensity','context_intensity_std','maximum_mask_depth','horizontal_fraction','vertical_fraction','compact_fraction','preserve_fraction','remove_fraction','ambiguous_fraction','reconstruction_dependency_fraction','component_pixels']
ROUTE_KEYS=['preserve','remove','ambiguous','horizontal','vertical','compact']

def newest_complete(root,required):
    hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
V75_RUN=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else newest_complete(V75_ROOT,['tables/cache_status.csv','tables/surrounding_area_evidence_by_component.csv','tables/surrounding_evidence_truth_diagnostics.csv'])
V77_RUN=Path(V77_RUN_OVERRIDE) if V77_RUN_OVERRIDE else newest_complete(V77_ROOT,['tables/asymmetric_component_dataset.csv','v7_7_manifest.json'])
if V75_RUN is None: raise RuntimeError('No complete V7.5 development run found. Set V75_RUN_OVERRIDE.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S');OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';DIFFS=OUT/'difference_images'
for d in [OUT,TABLES,DIFFS]: d.mkdir(parents=True,exist_ok=True)
print('V7.5 reference:',V75_RUN);print('V7.7 reference:',V77_RUN);print('Output:',OUT)


## Hash and input helpers


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def object_hash(value):return hashlib.sha256(json.dumps(value,sort_keys=True,default=str).encode()).hexdigest()
def image_map(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def array_digest(a):
    a=np.ascontiguousarray(a)
    return hashlib.sha256(a.tobytes()+str(a.shape).encode()+str(a.dtype).encode()).hexdigest()
def immutable(original,candidate,mask):
    out=original.copy();out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0])
    return out

def load_npz(path):
    with np.load(path) as z:return {k:z[k] for k in z.files}

clean_map=image_map(ROOT/'source'/'clean')
mask_map=image_map(ROOT/'masks'/'corrected')
cache_status=pd.read_csv(V75_RUN/'tables'/'cache_status.csv')
evidence=pd.read_csv(V75_RUN/'tables'/'surrounding_area_evidence_by_component.csv')
truth=pd.read_csv(V75_RUN/'tables'/'surrounding_evidence_truth_diagnostics.csv')
for frame_name,frame in [('evidence',evidence),('truth',truth)]:
    method_col=next((c for c in ['method','candidate','geometry_candidate'] if c in frame.columns),None)
    if method_col:frame.drop(frame[~frame[method_col].astype(str).str.lower().str.contains('t80')].index,inplace=True)
cache_lookup={str(r.page):Path(r.cache) for _,r in cache_status.iterrows()}
pages=sorted(set(cache_lookup)&set(clean_map)&set(mask_map))
if PAGE_ALLOWLIST is not None:pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
if MAX_PAGES is not None:pages=pages[:MAX_PAGES]
if not pages:raise RuntimeError('No matched development pages for audit.')
print('Pages selected:',len(pages))


## Frozen V7.8-builder implementation


In [ ]:
def builder_paper_value(observed,mask):
    ring=(cv2.dilate((mask>0).astype(np.uint8),np.ones((9,9),np.uint8),iterations=2)>0)&~(mask>0)
    values=observed[ring]
    return int(np.percentile(values,75)) if values.size else 255

def builder_geometry_support(observed,mask):
    visible=observed.copy();visible[mask>0]=255
    dark=(visible<DARK_THRESHOLD).astype(np.uint8)*255
    horizontal=cv2.morphologyEx(dark,cv2.MORPH_OPEN,np.ones((1,HORIZONTAL_KERNEL),np.uint8))>0
    vertical=cv2.morphologyEx(dark,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
    compact=(dark>0)&~(cv2.dilate(horizontal.astype(np.uint8),np.ones((3,5),np.uint8))>0)&~(cv2.dilate(vertical.astype(np.uint8),np.ones((5,3),np.uint8))>0)
    support=cv2.dilate((horizontal|vertical|compact).astype(np.uint8),np.ones((3,3),np.uint8))>0
    return horizontal,vertical,compact,support

def builder_geometry_t80(clean,mask):
    observed=clean.copy();observed[mask>0]=50
    horizontal,vertical,compact,support=builder_geometry_support(observed,mask)
    m=mask>0;paper=builder_paper_value(observed,mask)
    preserve=m&support
    remove=m&~support&(observed>=GEOMETRY_THRESHOLD)
    ambiguous=m&~preserve&~remove
    candidate=observed.copy();candidate[remove]=paper;candidate[ambiguous]=paper;candidate[preserve]=observed[preserve]
    corrected=immutable(observed,candidate,mask)
    return corrected,{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'observed':observed}

def builder_bbox_context(component,shape,margin=CONTEXT_MARGIN):
    ys,xs=np.where(component);x0,x1=xs.min(),xs.max()+1;y0,y1=ys.min(),ys.max()+1;h,w=shape
    return max(0,x0-margin),max(0,y0-margin),min(w,x1+margin),min(h,y1+margin),(x0,y0,x1,y1)

def builder_component_predictors(component_id,component,routing):
    observed=routing['observed'];x0,y0,x1,y1,bbox=builder_bbox_context(component,observed.shape);bx0,by0,bx1,by1=bbox
    context=np.zeros_like(component);context[y0:y1,x0:x1]=True;context&=~component
    vals=observed[context];edges=cv2.Canny(observed,EDGE_LOW,EDGE_HIGH)>0
    area=int(component.sum());w=bx1-bx0;h=by1-by0;dist=cv2.distanceTransform(component.astype(np.uint8),cv2.DIST_L2,5)
    def frac(key):return float(np.sum(routing[key]&component)/max(area,1))
    return {'component':component_id,'w':w,'h':h,'area':area,'aspect_ratio':w/max(h,1),'fill_ratio':area/max(w*h,1),'touches_page_border':bool(bx0==0 or by0==0 or bx1==observed.shape[1] or by1==observed.shape[0]),'context_pixels':int(context.sum()),'context_edge_density':float(edges[context].mean()) if context.any() else 0.0,'context_dark_density':float((vals<DARK_THRESHOLD).mean()) if vals.size else 0.0,'context_mean_intensity':float(vals.mean()) if vals.size else 255.0,'context_intensity_std':float(vals.std()) if vals.size else 0.0,'maximum_mask_depth':float(dist[component].max()) if area else 0.0,'horizontal_fraction':frac('horizontal'),'vertical_fraction':frac('vertical'),'compact_fraction':frac('compact'),'preserve_fraction':frac('preserve'),'remove_fraction':frac('remove'),'ambiguous_fraction':frac('ambiguous'),'reconstruction_dependency_fraction':frac('ambiguous'),'component_pixels':area}

BUILDER_SOURCE_HASH=object_hash({'geometry_support':inspect.getsource(builder_geometry_support),'geometry_t80':inspect.getsource(builder_geometry_t80),'component_predictors':inspect.getsource(builder_component_predictors),'settings':{'threshold':GEOMETRY_THRESHOLD,'context_margin':CONTEXT_MARGIN,'dark':DARK_THRESHOLD,'edges':[EDGE_LOW,EDGE_HIGH],'horizontal_kernel':HORIZONTAL_KERNEL,'minimum_component_area':MIN_COMPONENT_AREA}})
print('Builder implementation hash:',BUILDER_SOURCE_HASH)


## Frozen builder structural labels


In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def tolerant_scores(ref,pred,region,tol=1):
    ref=ref&region;pred=pred&region
    if not ref.any():return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0
    precision=float(np.sum(pred&rd)/max(pred.sum(),1));recall=float(np.sum(ref&pd)/ref.sum());f1=2*precision*recall/max(precision+recall,1e-12)
    return precision,recall,f1

def builder_truth(component,clean,corrected):
    ip,ir,if1=tolerant_scores(printed_ink(clean),printed_ink(corrected),component,1)
    ep,er,ef1=tolerant_scores(cv2.Canny(clean,EDGE_LOW,EDGE_HIGH)>0,cv2.Canny(corrected,EDGE_LOW,EDGE_HIGH)>0,component,1)
    reasons=[]
    if np.isfinite(ir) and ir<BUILDER_STRUCTURAL_THRESHOLDS['ink_recall_min']:reasons.append('ink_recall')
    if np.isfinite(if1) and if1<BUILDER_STRUCTURAL_THRESHOLDS['ink_f1_min']:reasons.append('ink_f1')
    if np.isfinite(ef1) and ef1<BUILDER_STRUCTURAL_THRESHOLDS['edge_f1_min']:reasons.append('edge_f1')
    return {'structural_safe':not reasons,'structural_failure_reasons':'|'.join(reasons),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1}


## Audit each page against the authoritative V7.5 cache and tables


In [ ]:
page_rows=[];route_rows=[];component_rows=[];predictor_rows=[];truth_rows=[];missing=[]
for index,page in enumerate(pages,1):
    clean=cv2.imread(str(clean_map[page]),0);mask_raw=cv2.imread(str(mask_map[page]),0);mask=np.where(mask_raw>0,255,0).astype(np.uint8)
    reference_path=cache_lookup[page]
    if not reference_path.exists():missing.append({'page':page,'artefact':'reference cache','path':str(reference_path)});continue
    ref=load_npz(reference_path)
    required=['mask','geometry_t80']+[f'g80_{k}' for k in ROUTE_KEYS]
    absent=[k for k in required if k not in ref]
    if absent:missing.append({'page':page,'artefact':'cache arrays','path':'|'.join(absent)});continue
    built,routes=builder_geometry_t80(clean,mask)
    ref_mask=ref['mask']>0;ref_img=ref['geometry_t80']
    page_rows.append({'page':page,'mask_equal':bool(np.array_equal(ref_mask,mask>0)),'reference_mask_pixels':int(ref_mask.sum()),'builder_mask_pixels':int((mask>0).sum()),'geometry_exact':bool(np.array_equal(ref_img,built)),'geometry_different_pixels':int(np.sum(ref_img!=built)),'geometry_max_abs_difference':int(np.max(np.abs(ref_img.astype(np.int16)-built.astype(np.int16)))),'reference_outside_mask_immutable':bool(np.array_equal(ref_img[ref_mask==0],clean[ref_mask==0])),'builder_outside_mask_immutable':bool(np.array_equal(built[mask==0],clean[mask==0])),'reference_geometry_hash':array_digest(ref_img),'builder_geometry_hash':array_digest(built)})
    for key in ROUTE_KEYS:
        a=ref[f'g80_{key}']>0;b=routes[key]
        route_rows.append({'page':page,'route':key,'exact':bool(np.array_equal(a,b)),'reference_pixels':int(a.sum()),'builder_pixels':int(b.sum()),'xor_pixels':int(np.logical_xor(a,b).sum()),'intersection_pixels':int((a&b).sum()),'union_pixels':int((a|b).sum()),'iou':float((a&b).sum()/max((a|b).sum(),1))})
    diff=np.abs(ref_img.astype(np.int16)-built.astype(np.int16)).astype(np.uint8)
    if np.any(diff):cv2.imwrite(str(DIFFS/f'{page}_geometry_absdiff.png'),diff)
    n_ref,lab_ref,stats_ref,_=cv2.connectedComponentsWithStats(ref_mask.astype(np.uint8),8)
    n_b,lab_b,stats_b,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    component_rows.append({'page':page,'reference_components_total':n_ref-1,'builder_components_total':n_b-1,'reference_components_ge20':int(sum(stats_ref[i,cv2.CC_STAT_AREA]>=MIN_COMPONENT_AREA for i in range(1,n_ref))),'builder_components_ge20':int(sum(stats_b[i,cv2.CC_STAT_AREA]>=MIN_COMPONENT_AREA for i in range(1,n_b))),'label_map_exact':bool(np.array_equal(lab_ref,lab_b))})
    ev_page=evidence[evidence.page.astype(str)==page].copy();tr_page=truth[truth.page.astype(str)==page].copy()
    for cid in range(1,n_b):
        if stats_b[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
        comp=lab_b==cid;pred=builder_component_predictors(cid,comp,routes);pred.update({'page':page})
        reference=ev_page[ev_page.component.astype(int)==cid]
        for field in PREDICTOR_FIELDS:
            rv=reference.iloc[0][field] if (len(reference)==1 and field in reference.columns) else np.nan;bv=pred[field]
            if isinstance(bv,(bool,np.bool_)):
                match=(bool(rv)==bool(bv)) if pd.notna(rv) else False;absdiff=0.0 if match else 1.0
            else:
                match=bool(np.isclose(float(rv),float(bv),atol=FLOAT_ABS_TOLERANCE,rtol=FLOAT_REL_TOLERANCE,equal_nan=True)) if pd.notna(rv) else False;absdiff=abs(float(rv)-float(bv)) if pd.notna(rv) else np.nan
            predictor_rows.append({'page':page,'component':cid,'field':field,'reference_value':rv,'builder_value':bv,'match':match,'absolute_difference':absdiff})
        bt=builder_truth(comp,clean,built);rt=tr_page[tr_page.component.astype(int)==cid]
        record={'page':page,'component':cid,**{f'builder_{k}':v for k,v in bt.items()}}
        for k in ['structural_safe','structural_failure_reasons','ink_precision','ink_recall','ink_f1','edge_precision','edge_recall','edge_f1']:
            record[f'reference_{k}']=rt.iloc[0][k] if len(rt)==1 and k in rt.columns else np.nan
        truth_rows.append(record)
    print(index,'/',len(pages),page)

page_df=pd.DataFrame(page_rows);route_df=pd.DataFrame(route_rows);component_df=pd.DataFrame(component_rows);predictor_df=pd.DataFrame(predictor_rows);truth_df=pd.DataFrame(truth_rows);missing_df=pd.DataFrame(missing)
for name,frame in [('page_equivalence',page_df),('route_equivalence',route_df),('component_equivalence',component_df),('predictor_equivalence',predictor_df),('truth_equivalence',truth_df),('missing_authoritative_artefacts',missing_df)]:frame.to_csv(TABLES/f'{name}.csv',index=False)


## Derive truth-label comparisons and fail-closed verdict


In [ ]:
def boolish(v):
    if pd.isna(v):return None
    if isinstance(v,(bool,np.bool_)):return bool(v)
    return str(v).strip().lower() in ['true','1','yes']
if not truth_df.empty:
    truth_df['structural_safe_match']=[boolish(a)==boolish(b) if boolish(a) is not None else False for a,b in zip(truth_df.reference_structural_safe,truth_df.builder_structural_safe)]
    truth_df['failure_reasons_match']=truth_df.reference_structural_failure_reasons.fillna('').astype(str)==truth_df.builder_structural_failure_reasons.fillna('').astype(str)
    metric_fields=['ink_precision','ink_recall','ink_f1','edge_precision','edge_recall','edge_f1']
    for f in metric_fields:
        truth_df[f'{f}_match']=np.isclose(pd.to_numeric(truth_df[f'reference_{f}'],errors='coerce'),pd.to_numeric(truth_df[f'builder_{f}'],errors='coerce'),atol=FLOAT_ABS_TOLERANCE,rtol=FLOAT_REL_TOLERANCE,equal_nan=True)
    truth_df.to_csv(TABLES/'truth_equivalence.csv',index=False)
checks={
 'authoritative_artefacts_complete':bool(missing_df.empty),
 'masks_exact':bool(len(page_df)>0 and page_df.mask_equal.all()),
 'geometry_exact':bool(len(page_df)>0 and page_df.geometry_exact.all()),
 'outside_mask_immutable_both':bool(len(page_df)>0 and page_df.reference_outside_mask_immutable.all() and page_df.builder_outside_mask_immutable.all()),
 'routing_exact':bool(len(route_df)>0 and route_df.exact.all()),
 'component_labels_exact':bool(len(component_df)>0 and component_df.label_map_exact.all()),
 'predictors_exact_within_tolerance':bool(len(predictor_df)>0 and predictor_df.match.all()),
 'structural_labels_exact':bool(len(truth_df)>0 and truth_df.structural_safe_match.all() and truth_df.failure_reasons_match.all()),
 'truth_metrics_exact_within_tolerance':bool(len(truth_df)>0 and all(truth_df[f'{f}_match'].all() for f in ['ink_precision','ink_recall','ink_f1','edge_precision','edge_recall','edge_f1'])) if len(truth_df) else False,
}
verdict='PASS' if all(checks.values()) else ('NOT_TESTED' if not checks['authoritative_artefacts_complete'] or not len(page_df) else 'FAIL')
summary=pd.DataFrame([{'verdict':verdict,**checks,'pages_audited':len(page_df),'components_audited':truth_df.shape[0],'geometry_different_pixels_total':int(page_df.geometry_different_pixels.sum()) if len(page_df) else np.nan,'routing_xor_pixels_total':int(route_df.xor_pixels.sum()) if len(route_df) else np.nan,'predictor_field_mismatches':int((~predictor_df.match).sum()) if len(predictor_df) else np.nan,'structural_label_mismatches':int((~truth_df.structural_safe_match).sum()) if len(truth_df) else np.nan}])
summary.to_csv(TABLES/'audit_verdict.csv',index=False);display(summary)


## Discrepancy ranking and review charts


In [ ]:
if len(predictor_df):
    pred_summary=predictor_df.groupby('field',as_index=False).agg(comparisons=('match','size'),mismatches=('match',lambda x:int((~x).sum())),maximum_absolute_difference=('absolute_difference','max'),mean_absolute_difference=('absolute_difference','mean')).sort_values(['mismatches','maximum_absolute_difference'],ascending=False)
else:pred_summary=pd.DataFrame()
pred_summary.to_csv(TABLES/'predictor_mismatch_summary.csv',index=False)
if len(route_df):
    rs=route_df.groupby('route',as_index=False).agg(pages=('page','nunique'),exact_pages=('exact','sum'),xor_pixels=('xor_pixels','sum'),mean_iou=('iou','mean'),minimum_iou=('iou','min')).sort_values('xor_pixels',ascending=False)
else:rs=pd.DataFrame()
rs.to_csv(TABLES/'route_mismatch_summary.csv',index=False)
if len(page_df):
    q=page_df.sort_values('geometry_different_pixels',ascending=False)
    plt.figure(figsize=(12,max(5,.25*len(q))));plt.barh(q.page,q.geometry_different_pixels);plt.xlabel('Different geometry_t80 pixels');plt.title('V7.8.1 geometry implementation differences by page');plt.tight_layout();plt.savefig(OUT/'geometry_difference_by_page.png',dpi=180);plt.close()
if len(rs):
    plt.figure(figsize=(9,5));plt.bar(rs.route,rs.xor_pixels);plt.ylabel('Total XOR pixels');plt.title('Routing-map differences by class');plt.tight_layout();plt.savefig(OUT/'routing_difference_by_class.png',dpi=180);plt.close()
if len(pred_summary):
    q=pred_summary.head(20).sort_values('mismatches');plt.figure(figsize=(10,7));plt.barh(q.field,q.mismatches);plt.xlabel('Component-field mismatches');plt.title('Predictor implementation mismatches');plt.tight_layout();plt.savefig(OUT/'predictor_mismatches.png',dpi=180);plt.close()


## Manifest and executive report


In [ ]:
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.8.1','purpose':'implementation equivalence audit','v7_5_reference_run':str(V75_RUN),'v7_7_reference_run':str(V77_RUN) if V77_RUN else None,'builder_source_hash':BUILDER_SOURCE_HASH,'pages_requested':len(pages),'pages_audited':len(page_df),'float_absolute_tolerance':FLOAT_ABS_TOLERANCE,'float_relative_tolerance':FLOAT_REL_TOLERANCE,'image_difference_tolerance':IMAGE_DIFFERENCE_TOLERANCE,'verdict':verdict,'checks':checks,'read_only_sources':True,'production_authorisation':False,'independent_validation_interpretation_allowed':bool(verdict=='PASS')}
(OUT/'v7_8_1_manifest.json').write_text(json.dumps(manifest,indent=2))
lines=['V7.8.1 IMPLEMENTATION-EQUIVALENCE AUDIT','',f'Verdict: {verdict}',f'Pages audited: {len(page_df)}',f'Components audited: {len(truth_df)}',f'Geometry pixel differences: {int(page_df.geometry_different_pixels.sum()) if len(page_df) else "not tested"}',f'Routing XOR pixels: {int(route_df.xor_pixels.sum()) if len(route_df) else "not tested"}',f'Predictor field mismatches: {int((~predictor_df.match).sum()) if len(predictor_df) else "not tested"}',f'Structural label mismatches: {int((~truth_df.structural_safe_match).sum()) if len(truth_df) else "not tested"}','','CHECKS']+[f'- {k}: {v}' for k,v in checks.items()]+['','A FAIL means the current V7.8 result must not be interpreted as a validation of V7.7. Fix implementation discrepancies without tuning against independent outcomes, then create a new locked release.','A PASS establishes computational equivalence on the audited development pages; it does not itself validate safety on independent scores.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines));print(chr(10).join(lines));print('Output:',OUT)
